# Lstar time series — hedge level, HRs, and residual returns

**[Get API key](https://riskmodels.app/get-key)** · **[Open in Colab](https://colab.research.google.com/github/BlueWaterCorp/RiskModels_API/blob/main/sdk/notebooks/lstar_timeseries.ipynb)** · `GET /api/lstar` · **$0.02/request**

For each trading day the API picks the **simplest** hedge level whose marginal explained return clears a threshold (default **1%**):

| Level | Hedge stack |
|-------|-------------|
| **L1** | market ETF only (`market_hr`; sector/subsector HRs are null) |
| **L2** | market + sector (`market_hr`, `sector_hr`) |
| **L3** | market + sector + subsector (all three HRs) |

**Conventions:** HR = dollars of ETF per **$1** of stock. `total_er` and marginal ERs are **variance shares** in \([0,1]\). `residual_return` is the **daily simple return** at the chosen level (not ER).

Requires **`riskmodels-py>=0.3.6`** (`client.get_lstar()`). Set `RISKMODELS_API_KEY` in `.env.local` or your shell.


### Colab only (skip locally)


In [ ]:
import sys

try:
    import google.colab  # noqa: F401

    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    import subprocess

    _deps = ["requests", "python-dotenv"]
    _pypi = "riskmodels-py[viz]>=0.3.6,<0.4"
    _git = (
        "riskmodels-py[viz] @ git+https://github.com/BlueWaterCorp/RiskModels_API.git"
        "@main#subdirectory=sdk"
    )
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", _pypi, *_deps],
            stdout=subprocess.DEVNULL,
        )
        print("Colab: installed riskmodels-py[viz] from PyPI.")
    except subprocess.CalledProcessError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", _git, *_deps],
            stdout=subprocess.DEVNULL,
        )
        print("Colab: installed riskmodels-py[viz] from GitHub main.")
else:
    print("Local: use your venv (pip install riskmodels-py[viz]>=0.3.6).")


## Parameters & connect

Override with env vars: `LSTAR_TICKER`, `LSTAR_YEARS`, `LSTAR_THRESHOLD`.


In [ ]:
import os

import pandas as pd
from IPython.display import Markdown, display

from riskmodels import RiskModelsClient
from riskmodels.notebook import ensure_riskmodels_api_key

ensure_riskmodels_api_key()
client = RiskModelsClient.from_env()

TICKER = os.environ.get("LSTAR_TICKER", "NFLX").upper()
YEARS = int(os.environ.get("LSTAR_YEARS", "5"))
THRESHOLD = float(os.environ.get("LSTAR_THRESHOLD", "0.01"))

display(
    Markdown(
        f"**Ticker:** `{TICKER}` · **Years:** {YEARS} · **Threshold:** {THRESHOLD:.2%}"
    )
)


## Fetch Lstar series


In [ ]:
raw = client.get_lstar(TICKER, years=YEARS, threshold=THRESHOLD)
threshold_used = raw.attrs.get("threshold_used", THRESHOLD)

if raw.empty:
    raise RuntimeError(f"No Lstar rows returned for {TICKER}.")

ts = raw.copy()
ts["date"] = pd.to_datetime(ts["date"])
ts = ts.sort_values("date").reset_index(drop=True)

# Human-readable hedge legs (ETF tickers come from metrics; HRs are $/$ here)
def _fmt_hr(v):
    return None if v is None or pd.isna(v) else float(v)


def _hedge_summary(row) -> str:
    parts = []
    m = _fmt_hr(row["market_hr"])
    s = _fmt_hr(row["sector_hr"])
    u = _fmt_hr(row["subsector_hr"])
    if m is not None:
        parts.append(f"mkt {m:+.3f}")
    if s is not None:
        parts.append(f"sec {s:+.3f}")
    if u is not None:
        parts.append(f"sub {u:+.3f}")
    return " · ".join(parts) if parts else "—"


ts["hedge_stack"] = ts.apply(_hedge_summary, axis=1)
ts["total_er_pct"] = ts["total_er"] * 100.0
ts["residual_bps"] = ts["residual_return"] * 10_000.0
ts["l2_sector_er_pct"] = ts["l2_sector_er"] * 100.0
ts["l3_subsector_er_pct"] = ts["l3_subsector_er"] * 100.0
ts["cum_residual"] = ts["residual_return"].fillna(0.0).cumsum()

display_cols = [
    "date",
    "lstar",
    "market_hr",
    "sector_hr",
    "subsector_hr",
    "total_er_pct",
    "residual_bps",
    "l2_sector_er_pct",
    "l3_subsector_er_pct",
    "hedge_stack",
    "cum_residual",
]
lstar_df = ts[display_cols].set_index("date")

level_counts = ts["lstar"].value_counts(dropna=False).reindex(["L1", "L2", "L3"]).fillna(0).astype(int)
n_days = len(ts)
pct = (level_counts / n_days * 100).round(1)

summary = pd.DataFrame(
    {
        "days": level_counts,
        "share_pct": pct,
    }
)

display(
    Markdown(
        f"**{TICKER}** · {n_days:,} trading days · threshold used **{float(threshold_used):.2%}**"
    )
)
display(Markdown("### Lstar level mix"))
display(summary)
print(raw.attrs.get("legend", ""))


## Time series table (latest 10 rows)

Full frame is in **`lstar_df`** (`date` index). Percent columns are display-only (`total_er_pct`, …).


In [ ]:
styled = (
    lstar_df.tail(10)
    .style.format(
        {
            "market_hr": "{:+.4f}",
            "sector_hr": "{:+.4f}",
            "subsector_hr": "{:+.4f}",
            "total_er_pct": "{:.2f}%",
            "residual_bps": "{:+.1f}",
            "l2_sector_er_pct": "{:.2f}%",
            "l3_subsector_er_pct": "{:.2f}%",
            "cum_residual": "{:+.2%}",
        },
        na_rep="—",
    )
    .set_caption(f"{TICKER} Lstar — last 10 sessions")
)
display(styled)

# Raw numeric columns (API shape) for export / further analysis
api_df = ts[
    [
        "date",
        "lstar",
        "market_hr",
        "sector_hr",
        "subsector_hr",
        "total_er",
        "residual_return",
        "l2_sector_er",
        "l3_subsector_er",
    ]
].set_index("date")
api_df.head(3)


## Charts (optional — needs `riskmodels-py[viz]`)


In [ ]:
try:
    import plotly.express as px
except ImportError:
    print("Install plotly: pip install 'riskmodels-py[viz]'")
else:
    plot_df = ts.dropna(subset=["lstar"]).copy()
    plot_df["date_str"] = plot_df["date"].dt.strftime("%Y-%m-%d")

    fig_levels = px.scatter(
        plot_df,
        x="date",
        y="lstar",
        color="lstar",
        category_orders={"lstar": ["L1", "L2", "L3"]},
        title=f"{TICKER} recommended hedge level (Lstar)",
        labels={"lstar": "Lstar", "date": "Date"},
    )
    fig_levels.update_layout(showlegend=False, height=320)
    fig_levels.show()

    fig_cum = px.line(
        plot_df,
        x="date",
        y="cum_residual",
        title=f"{TICKER} cumulative residual return at chosen Lstar level",
        labels={"cum_residual": "Cumulative residual (simple)", "date": "Date"},
    )
    fig_cum.update_layout(height=320)
    fig_cum.show()


## Export

```python
# Parquet / CSV from the API-shaped frame
api_df.to_parquet("lstar_nflx.parquet")
api_df.to_csv("lstar_nflx.csv")
```
